# Doctor Assistant — lung-nodule detection + MedSAM2 Colab server

Runs automatic MONAI lung-nodule candidate detection and MedSAM2 candidate/manual-box refinement on one Colab GPU, exposed through ngrok. The Mac keeps OHIF/Orthanc only. Use an **L4 or Ampere-or-newer GPU with at least 18 GiB**. A shared lock permits one heavy inference at a time.

In [ ]:
# Configuration — no credentials belong in this cell.
from pathlib import Path

REPO_URL = 'https://github.com/Hamza09Hamza/doctor_assistant.git'
REPO_REF = 'main'
REPO_DIR = Path('/content/doctor_assistant')
MEDSAM2_DIR = Path('/content/MedSAM2')
LUNG_NODULE_BUNDLE_ROOT = Path('/content/doctor_assistant_models')
MEDSAM2_CODE_COMMIT = '332f30d420f1d1b08e2a79b3ae6a602458808383'
MEDSAM2_MODEL_REVISION = 'c0b302de5e19211e9f1425c6215f6172eb5eca7f'
MEDSAM2_CT_SHA256 = '78f7e125418dfd6fec22f4afe90bcd85cb1d4423d0a9df36f7a87ed63aa1a5f5'
MONAI_DETECTOR_BUNDLE_VERSION = '0.6.10'
MONAI_DETECTOR_SHA256 = 'b5e79231466adae93a6fe8e8594029e9add142914e223b879aa0343bb2402d01'
API_PORT = 8000
NGROK_TOKEN = '3ERc2GB0MMOjOt2dleZiNurkxZl_55C3LUxsrrK7hH8qFtBx7'
DOWNLOAD_ORTHANC_MACOS = False  # Set True only if the Mac needs the FortiGuard-safe relay.


In [ ]:
# Fail before setup if this runtime cannot safely host the model.
import psutil
import torch

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU.'
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 1024**3
system_gib = psutil.virtual_memory().total / 1024**3
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU: {gpu.name}, compute capability {major}.{minor}, {gpu_gib:.1f} GiB')
print(f'System RAM: {system_gib:.1f} GiB')
assert major >= 8, 'Use L4/A100/Ampere-or-newer: the API uses CUDA bfloat16 autocast.'
assert gpu_gib >= 18, 'At least 18 GiB GPU memory is required for this bounded test.'
assert system_gib >= 24, 'At least 24 GiB system RAM is required for detector CPU stitching.'


In [ ]:
# Clone the pushed application and install only the server/runtime dependencies.
import os
import subprocess
import sys

def run(command, *, cwd=None, env=None):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, cwd=cwd, env=env, check=True)

if REPO_DIR.exists():
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', REPO_REF], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO_DIR)
else:
    run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)])

run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'pip'])
run([sys.executable, '-m', 'pip', 'install', '--quiet',
     '-r', str(REPO_DIR / 'requirements-api.txt'),
     'idc-index==0.12.5', 'scipy>=1.12', 'SimpleITK>=2.4', 'itk>=5.4,<6',
     'monai==1.6.0', 'huggingface-hub>=0.23,<2', 'pyngrok>=7.2'])

if not MEDSAM2_DIR.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/bowang-lab/MedSAM2.git', str(MEDSAM2_DIR)])
run(['git', 'fetch', '--depth', '1', 'origin', MEDSAM2_CODE_COMMIT], cwd=MEDSAM2_DIR)
run(['git', 'checkout', '--detach', MEDSAM2_CODE_COMMIT], cwd=MEDSAM2_DIR)
install_env = dict(os.environ, SAM2_BUILD_CUDA='0')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(MEDSAM2_DIR)], env=install_env)
# Editable installs add a .pth hook that this already-running notebook kernel
# does not reprocess. Probe imports in a fresh interpreter, exactly like Uvicorn.
runtime_versions = subprocess.check_output([
    sys.executable, '-c',
    "import itk, monai, SimpleITK, sam2; "
    "print({'monai': monai.__version__, 'itk': itk.Version.GetITKVersion(), "
    "'SimpleITK': SimpleITK.Version_VersionString(), 'sam2': sam2.__file__})"
], text=True).strip()
print('Application commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip())
print('MedSAM2 commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=MEDSAM2_DIR, text=True).strip())
print('Runtime imports:', runtime_versions)


In [ ]:
# Download the two task-specific model artifacts, then stage the pinned detector demo.
import hashlib
import json
from huggingface_hub import hf_hub_download
from monai.bundle import download as download_monai_bundle

def verify_sha256(path, expected):
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'SHA-256 mismatch for {path}: {actual}')

checkpoint_dir = MEDSAM2_DIR / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint = checkpoint_dir / 'MedSAM2_CTLesion.pt'
if checkpoint.exists():
    try:
        verify_sha256(checkpoint, MEDSAM2_CT_SHA256)
    except RuntimeError:
        checkpoint.unlink()
if not checkpoint.exists():
    downloaded = Path(hf_hub_download(
        repo_id='wanglab/MedSAM2',
        filename='MedSAM2_CTLesion.pt',
        revision=MEDSAM2_MODEL_REVISION,
    ))
    checkpoint.write_bytes(downloaded.read_bytes())
verify_sha256(checkpoint, MEDSAM2_CT_SHA256)
print(f'Checkpoint: {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MiB)')

detector_checkpoint = (
    LUNG_NODULE_BUNDLE_ROOT / 'lung_nodule_ct_detection/models/model.pt'
)
if not detector_checkpoint.is_file():
    LUNG_NODULE_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
    print('Downloading MONAI lung-nodule detector bundle (first run only)...', flush=True)
    download_monai_bundle(
        name='lung_nodule_ct_detection',
        version=MONAI_DETECTOR_BUNDLE_VERSION,
        bundle_dir=str(LUNG_NODULE_BUNDLE_ROOT),
    )
assert detector_checkpoint.is_file(), detector_checkpoint
verify_sha256(detector_checkpoint, MONAI_DETECTOR_SHA256)
print(f'Detector checkpoint: {detector_checkpoint} ({detector_checkpoint.stat().st_size / 1024**2:.1f} MiB)')

run([sys.executable, 'scripts/prepare_lidc_nodule_detector_demo.py'], cwd=REPO_DIR)
manifest_path = REPO_DIR / 'data/validation/lidc_idri_0117_nodule_detector/nodule_detector_demo_manifest.json'
manifest = json.loads(manifest_path.read_text())
print('Remote API series ID:', manifest['api_series_id'])
print('DICOM Series UID:', manifest['series_instance_uid'])

# Download once in Colab; the FastAPI/ngrok route serves it directly to the Mac.
if DOWNLOAD_ORTHANC_MACOS:
    orthanc_url = 'https://orthanc.uclouvain.be/downloads/macos/packages/universal/Orthanc-macOS-26.4.2.zip'
    orthanc_zip = Path('/content/Orthanc-macOS-26.4.2.zip')
    if not orthanc_zip.exists():
        print('Downloading official Orthanc macOS package in Colab (~320 MB)...', flush=True)
        import requests
        with requests.get(orthanc_url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with orthanc_zip.open('wb') as output:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        output.write(chunk)
    assert orthanc_zip.stat().st_size > 300 * 1024**2, 'Orthanc download is unexpectedly small.'
    print(f'Orthanc relay ready: {orthanc_zip} ({orthanc_zip.stat().st_size / 1024**2:.1f} MiB)')


In [ ]:
# Start one API worker, verify it locally, then open the ngrok HTTPS tunnel.
import os
import subprocess
import sys
import time
import requests
from pyngrok import ngrok

try:
    SERVER.terminate()
    SERVER.wait(timeout=10)
except NameError:
    pass
except subprocess.TimeoutExpired:
    SERVER.kill()
    SERVER.wait(timeout=10)
except Exception:
    pass
ngrok.kill()

server_env = dict(os.environ)
server_env['PYTHONPATH'] = os.pathsep.join(
    [str(REPO_DIR), str(MEDSAM2_DIR), server_env.get('PYTHONPATH', '')]
)
server_env.update({
    'DATABASE_URL': manifest['database_url'],
    'STORAGE_DIR': manifest['storage_dir'],
    'MEDSAM2_BACKEND': 'torch',
    'MEDSAM2_CHECKPOINT_PATH': str(checkpoint),
    'MEDSAM2_MODEL_CONFIG': 'configs/sam2.1_hiera_t512.yaml',
    'MEDSAM2_DEVICE': 'cuda',
    'LUNG_NODULE_BUNDLE_ROOT': str(LUNG_NODULE_BUNDLE_ROOT),
    'COLAB_PRELOAD_MODELS': '1',
    'OMP_NUM_THREADS': '2',
    'MKL_NUM_THREADS': '2',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'COLAB_DOWNLOAD_DIRECTORY': '/content',
})
log_path = Path('/content/doctor_assistant_colab_api.log')
SERVER_LOG = log_path.open('w')
SERVER = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.colab_server:create_app', '--factory',
     '--host', '127.0.0.1', '--port', str(API_PORT), '--workers', '1',
     '--timeout-keep-alive', '300'],
    cwd=REPO_DIR, env=server_env, stdout=SERVER_LOG, stderr=subprocess.STDOUT,
)
for _ in range(60):
    if SERVER.poll() is not None:
        SERVER_LOG.flush()
        raise RuntimeError(log_path.read_text()[-6000:])
    try:
        local_health = requests.get(f'http://127.0.0.1:{API_PORT}/health', timeout=2)
        if local_health.ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError('API did not become healthy within 60 seconds')
health = local_health.json()
assert health['medsam2_configured'], health
assert health['lung_nodule_detector_configured'], health
assert health['ready'], health

ngrok.set_auth_token(NGROK_TOKEN)
TUNNEL = ngrok.connect(addr=API_PORT, proto='http')
PUBLIC_API_URL = TUNNEL.public_url.replace('http://', 'https://')
public_health = requests.get(
    f'{PUBLIC_API_URL}/health',
    headers={'ngrok-skip-browser-warning': 'doctor-assistant'},
    timeout=20,
)
public_health.raise_for_status()
assert public_health.json()['ready'], public_health.json()
print('\nCOLAB API READY:', PUBLIC_API_URL)
print('Health:', public_health.json())
if DOWNLOAD_ORTHANC_MACOS:
    print('\nRun this on the Mac to fetch Orthanc through this same ngrok tunnel:')
    print(f"ORTHANC_ARCHIVE_URL='{PUBLIC_API_URL}/downloads/orthanc-macos' bash scripts/install_orthanc_macos.sh")
print('\nRun these in separate Mac terminals:')
print('A. bash scripts/start_orthanc_macos.sh')
print('B. source .venv-mlx/bin/activate && python scripts/prepare_lidc_nodule_detector_demo.py --upload-orthanc')
print(f"C. bash scripts/start_ohif_with_remote_api.sh '{PUBLIC_API_URL}'")
print('Open the Clinique Amina LIDC-IDRI-0117 URL printed by command B.')


In [ ]:
# Keep the interactive runtime active and show bounded status. Interrupt this cell to stop monitoring.
import time
print('Monitoring one-worker API. Interrupt this cell when the test is finished.')
while SERVER.poll() is None:
    rss_gib = psutil.Process(SERVER.pid).memory_info().rss / 1024**3
    gpu_usage = subprocess.run(
        ['nvidia-smi', '--query-compute-apps=pid,used_memory', '--format=csv,noheader,nounits'],
        capture_output=True, text=True, check=False,
    ).stdout.strip() or 'no CUDA process allocation yet'
    print(f'API alive | server RSS {rss_gib:.2f} GiB | GPU processes [pid, MiB]: {gpu_usage}', flush=True)
    time.sleep(30)
print('API process exited. Tail of log:')
print(log_path.read_text()[-6000:])


In [ ]:
# Cleanup after interrupting the monitoring cell.
ngrok.kill()
if SERVER.poll() is None:
    SERVER.terminate()
    try:
        SERVER.wait(timeout=15)
    except subprocess.TimeoutExpired:
        SERVER.kill()
SERVER_LOG.close()
torch.cuda.empty_cache()
print('ngrok tunnel closed and API process stopped.')
